In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CBTDI10", con=connection2)

connection2.close()



In [2]:
base2

,tipdocidenpercod,tipdocidenpernom,tipdocidenperdescor,tipdocidenperequicod,tipdocidenestregcod
0,1,D.N.I.,D.N.I.,1,1
1,2,CARNE DE EXTRANJERIA/PASAPORTE,C.E./PAS,2,1
2,3,PASAPORTE,PASAPORT,3,0
3,8,NEONATO,NEONATO,8,1
4,6,CARNET DE IDENTIDAD - RR.EE.,C.I.RREE,Q,1
5,7,CARNET PERMISO TEMP PERM CPP,CPP,S,1
6,9,NO UTILIZAR,NO USAR,O,0
7,4,DOCUMENTO IDENTIDAD EXTRANJERO,D.I.EXT.,R,1
8,5,PERMISO TEMPORAL DE PERMANENCIA,PTP,P,1
9,X,NO.DOC,NO.DOC,None,0


In [3]:
a=base2[base2.duplicated(subset=["tipdocidenpercod"])]
a

,tipdocidenpercod,tipdocidenpernom,tipdocidenperdescor,tipdocidenperequicod,tipdocidenestregcod


In [4]:
base2.columns

Index(['tipdocidenpercod', 'tipdocidenpernom', 'tipdocidenperdescor',
       'tipdocidenperequicod', 'tipdocidenestregcod'],
      dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_cbtdi10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbtdi10 FALSO CON LO OBTENIDO DEL ORACLE
query_count_before = "SELECT COUNT(*) FROM cbtdi10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cbtdi10 antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_cbtdi10 
ALTER COLUMN tipdocidenpercod TYPE CHARACTER (1),
ALTER COLUMN tipdocidenpernom TYPE CHARACTER (35),
ALTER COLUMN tipdocidenperdescor TYPE CHARACTER (8),
ALTER COLUMN tipdocidenperequicod TYPE CHARACTER (1),
ALTER COLUMN tipdocidenestregcod TYPE CHARACTER (1);


UPDATE cbtdi10 
SET 

tipdocidenpernom= t.tipdocidenpernom,
tipdocidenperdescor= t.tipdocidenperdescor,
tipdocidenperequicod= t.tipdocidenperequicod,
tipdocidenestregcod= t.tipdocidenestregcod




FROM tmp_cbtdi10 t 
WHERE cbtdi10.tipdocidenpercod = t.tipdocidenpercod and cbtdi10.tipdocidenpercod IS NOT NULL and t.tipdocidenpercod IS NOT NULL ;

INSERT INTO cbtdi10 
(tipdocidenpercod, tipdocidenpernom, tipdocidenperdescor,tipdocidenperequicod, tipdocidenestregcod) 

SELECT 
tipdocidenpercod, tipdocidenpernom, tipdocidenperdescor,tipdocidenperequicod, tipdocidenestregcod

FROM tmp_cbtdi10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cbtdi10 
    WHERE cbtdi10.tipdocidenpercod = tmp_cbtdi10.tipdocidenpercod and cbtdi10.tipdocidenpercod IS NOT NULL and tmp_cbtdi10.tipdocidenpercod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cbtdi10;
DELETE FROM cbtdi10 WHERE tipdocidenpercod ='';
SELECT COUNT(*) FROM cbtdi10;

"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla cbtdi10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

base2 = pd.read_sql_query(f"SELECT * FROM cbtdi10", con=connection3)

connection3.close()


Cantidad de filas en la tabla cbtdi10 antes de la inserción: 11
Cantidad de filas en la tabla cbtdi10 después de la inserción: 11
La cantidad de filas insertadas fue de: 0


In [6]:
base2.columns

Index(['tipdocidenpercod', 'tipdocidenpernom', 'tipdocidenperdescor',
       'tipdocidenperequicod', 'tipdocidenestregcod'],
      dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD_2"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cbtdi10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_tipdoc"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_tipdoc antes de la inserción: {cant_antes}")


query="""

ALTER TABLE tmp_cbtdi10 
ALTER COLUMN tipdocidenpercod TYPE CHARACTER (1),
ALTER COLUMN tipdocidenpernom TYPE CHARACTER (35),
ALTER COLUMN tipdocidenperdescor TYPE CHARACTER (8),
ALTER COLUMN tipdocidenperequicod TYPE CHARACTER (1),
ALTER COLUMN tipdocidenestregcod TYPE CHARACTER (1);


INSERT INTO dim_tipdoc (cod_tdo, des_tdo,abr_tdo) 
SELECT tipdocidenpercod, tipdocidenpernom,  tipdocidenperdescor

FROM tmp_cbtdi10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_tipdoc 
    WHERE dim_tipdoc.cod_tdo = tmp_cbtdi10.tipdocidenpercod
);

DROP TABLE tmp_cbtdi10;

SELECT COUNT(*) FROM dim_tipdoc;
"""

c1 = text(query)
cursor = connection4.execute(c1)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_tipdoc después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection4.close()

Cantidad de filas en la tabla dim_tipdoc antes de la inserción: 0
Cantidad de filas en la tabla dim_tipdoc después de la inserción: 11
La cantidad de filas insertadas fue de: 11
